# Combined Model Full-Track Waterfall

Reads the full I/Q track for a test-set event, applies the pretrained
`LitS4CombinedModel` (denoising + regression) in non-overlapping windows
of `cutoff` samples, and produces a time vs. frequency waterfall plot
coloured by scaled signal-to-noise ratio.

The training config is loaded automatically from the run directory that
contains the checkpoint — no separate `CONFIG_PATH` is needed.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import h5py
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq, fftshift

from src.models.model import LitS4CombinedModel
from src.models.networks import S4DCombinedModel
from src.utils.noise import noise_model, bandpass_filter

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Configuration

Only the checkpoint path is required — the training config is discovered
automatically from the same Lightning run directory.

In [ ]:
# ── Only this path needs to be set ───────────────────────────────────────────
CHECKPOINT_PATH = '../runs/combined_denoise_regression/lightning_logs/<version>/checkpoints/last.ckpt'

# Index of the test-set track to visualise (0 = first test track)
TEST_TRACK_IDX = 0

# Maximum number of segments to process (None = all that fit in the track)
MAX_SEGMENTS = None

# Baseband half-width to display around the carrier frequency [Hz]
DISPLAY_BW_HZ = 6e6   # ±6 MHz → 12 MHz total display bandwidth

# Sampling rate (matches noise_model in src/utils/noise.py)
FS = 403e6   # Hz
# ─────────────────────────────────────────────────────────────────────────────

# Discover config.yaml saved by LightningCLI alongside the checkpoint
# Layout:  <run_dir>/lightning_logs/<version>/checkpoints/last.ckpt
#          <run_dir>/lightning_logs/<version>/config.yaml
ckpt_path   = Path(CHECKPOINT_PATH)
config_path = ckpt_path.parent.parent / 'config.yaml'

if not config_path.exists():
    raise FileNotFoundError(
        f'Expected config at {config_path}.\n'
        'LightningCLI saves config.yaml next to the checkpoints directory.'
    )

print(f'Checkpoint : {ckpt_path}')
print(f'Config     : {config_path}')

## 2. Load Config, Build Encoder, Load Model

In [ ]:
with open(config_path) as f:
    config = yaml.safe_load(f)

data_cfg  = config['data']['init_args']
model_cfg = config['model']['init_args']
enc_cfg   = model_cfg['encoder']['init_args']

INPUTS       = data_cfg['inputs']                  # ['output_ts_I', 'output_ts_Q']
DATA_DIR     = data_cfg['data_dir']
CUTOFF       = data_cfg.get('cutoff', 8192)
NOISE_CONST  = data_cfg.get('noise_const', 1.0)
APPLY_FILTER = data_cfg.get('apply_filter', False)

print(f'Data dir      : {DATA_DIR}')
print(f'Inputs        : {INPUTS}')
print(f'Cutoff        : {CUTOFF} samples  ({CUTOFF / FS * 1e6:.1f} µs per segment)')
print(f'Noise const   : {NOISE_CONST}')
print(f'Apply filter  : {APPLY_FILTER}')

In [ ]:
encoder = S4DCombinedModel(
    d_input                         = enc_cfg['d_input'],
    d_output                        = enc_cfg['d_output'],
    denoiser_d_model                = enc_cfg.get('denoiser_d_model', 128),
    denoiser_n_layers               = enc_cfg.get('denoiser_n_layers', 6),
    denoiser_dropout                = enc_cfg.get('denoiser_dropout', 0.0),
    denoiser_prenorm                = enc_cfg.get('denoiser_prenorm', False),
    denoiser_gradient_checkpointing = enc_cfg.get('denoiser_gradient_checkpointing', False),
    regressor_d_model               = enc_cfg.get('regressor_d_model', 128),
    regressor_n_layers              = enc_cfg.get('regressor_n_layers', 6),
    regressor_dropout               = enc_cfg.get('regressor_dropout', 0.0),
    regressor_prenorm               = enc_cfg.get('regressor_prenorm', False),
    regressor_fc_hidden             = enc_cfg.get('regressor_fc_hidden', [64, 32]),
    regressor_gradient_checkpointing= enc_cfg.get('regressor_gradient_checkpointing', False),
)

model = LitS4CombinedModel.load_from_checkpoint(
    str(ckpt_path),
    encoder=encoder,
    map_location=device,
).to(device).eval()

print('Model loaded successfully')
print(f'  Denoiser : d_model={enc_cfg.get("denoiser_d_model", 128)}, '
      f'n_layers={enc_cfg.get("denoiser_n_layers", 6)}')
print(f'  Regressor: d_model={enc_cfg.get("regressor_d_model", 128)}, '
      f'n_layers={enc_cfg.get("regressor_n_layers", 6)}')

## 3. Identify the Test-Set Track

Reproduce the exact 80/10/10 split (seed 42) used during training so that
we evaluate on a track the model has never seen.

In [ ]:
hdf5_files = sorted([
    os.path.join(DATA_DIR, fn)
    for fn in os.listdir(DATA_DIR)
    if fn.endswith('.hdf5') or fn.endswith('.h5')
])

# Build global index (file_path, local_row_idx) matching LitCombinedDataModule
global_index = []
for path in hdf5_files:
    with h5py.File(path, 'r') as f:
        n = f[INPUTS[0]].shape[0]
    global_index.extend((path, i) for i in range(n))

total_tracks = len(global_index)
print(f'HDF5 files            : {len(hdf5_files)}')
print(f'Total tracks          : {total_tracks}')

# Replicate the training split (same seed / ratios as LitCombinedDataModule)
rng_split = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    range(total_tracks), [0.8, 0.1, 0.1], generator=rng_split
)
test_indices = list(test_ds.indices)
print(f'Test-set size         : {len(test_indices)} tracks')

# Resolve the requested test track to its HDF5 file and local row
global_row = test_indices[TEST_TRACK_IDX]
hdf5_path, local_row = global_index[global_row]
print(f'\nTest track {TEST_TRACK_IDX}  →  {os.path.basename(hdf5_path)}, row {local_row}')

## 4. Read the Full Track

In [ ]:
with h5py.File(hdf5_path, 'r') as f:
    track_I = f[INPUTS[0]][local_row][:].astype(np.float32)  # full I channel
    track_Q = f[INPUTS[1]][local_row][:].astype(np.float32)  # full Q channel

    # Carrier frequency for the physical frequency axis
    if 'avg_carrier_frequency_Hz' in f:
        carrier_freq_hz = float(f['avg_carrier_frequency_Hz'][local_row])
    else:
        carrier_freq_hz = 0.0   # fall back to baseband-only axis

track_len        = len(track_I)
n_segments       = track_len // CUTOFF
if MAX_SEGMENTS is not None:
    n_segments = min(n_segments, MAX_SEGMENTS)

print(f'Track length  : {track_len:,} samples  ({track_len / FS * 1e3:.3f} ms)')
print(f'Segments      : {n_segments}  ×  {CUTOFF} samples  ({CUTOFF / FS * 1e6:.1f} µs each)')
print(f'Carrier freq  : {carrier_freq_hz / 1e9:.6f} GHz')

## 5. Apply Combined Model Segment by Segment

`LitS4CombinedModel.forward(x)` returns `(x_denoised, y_pred)`. We keep
the denoised I/Q signal and discard the regression output for this plot.

In [ ]:
denoised_I = np.zeros(n_segments * CUTOFF, dtype=np.float32)
denoised_Q = np.zeros(n_segments * CUTOFF, dtype=np.float32)

for s in range(n_segments):
    start = s * CUTOFF
    end   = start + CUTOFF

    seg_I = track_I[start:end].copy()
    seg_Q = track_Q[start:end].copy()

    # Add noise + normalise (mirrors Project8SimCombined.__getitem__)
    X_noisy = np.stack([seg_I, seg_Q], axis=-1)   # (cutoff, 2)
    X_norm  = np.zeros_like(X_noisy)
    norms   = np.zeros(2, dtype=np.float32)

    for j in range(2):
        noise = noise_model(CUTOFF, NOISE_CONST)
        Xn    = X_noisy[:, j] + noise
        if APPLY_FILTER:
            Xn = bandpass_filter(Xn)
        s_std        = float(np.std(Xn)) + 1e-8
        norms[j]     = s_std
        X_norm[:, j] = Xn / s_std

    # Run the combined model; forward returns (x_denoised, y_pred)
    x_tensor = torch.tensor(X_norm, dtype=torch.float32).unsqueeze(0).to(device)  # (1, L, 2)
    with torch.no_grad():
        x_denoised_norm, _y_pred = model(x_tensor)           # (1, L, 2)
    x_denoised_norm = x_denoised_norm.squeeze(0).cpu().numpy()  # (L, 2)

    # Denormalise back to original amplitude scale
    denoised_I[start:end] = x_denoised_norm[:, 0] * norms[0]
    denoised_Q[start:end] = x_denoised_norm[:, 1] * norms[1]

    if (s + 1) % max(1, n_segments // 10) == 0:
        print(f'  processed {s+1:4d} / {n_segments} segments')

print('Done.')

## 6. Compute Power Spectra

Treat each segment as a complex sample `I + jQ`, compute the FFT power
spectrum, and normalise each column by its median (noise floor estimate).

In [ ]:
# Baseband frequency axis (shifted so DC is in the centre)
baseband_freqs_hz  = fftshift(fftfreq(CUTOFF, d=1.0 / FS))          # Hz
physical_freqs_ghz = (carrier_freq_hz + baseband_freqs_hz) / 1e9    # GHz

# Select a narrow display band around the carrier
freq_mask = np.abs(baseband_freqs_hz) <= DISPLAY_BW_HZ

# Time centre of each segment
seg_dt_s    = CUTOFF / FS
seg_times_s = np.arange(n_segments) * seg_dt_s + seg_dt_s / 2

# 2-D power array: shape (n_display_freqs, n_segments)
n_display = int(freq_mask.sum())
power_map = np.zeros((n_display, n_segments), dtype=np.float32)

for s in range(n_segments):
    start    = s * CUTOFF
    end      = start + CUTOFF
    cplx     = denoised_I[start:end] + 1j * denoised_Q[start:end]
    spectrum = fftshift(np.abs(fft(cplx)) ** 2)
    power_map[:, s] = spectrum[freq_mask]

# Per-segment noise floor = median power across the displayed frequency bins
noise_floor    = np.median(power_map, axis=0, keepdims=True)   # (1, n_segments)
noise_floor    = np.where(noise_floor > 0, noise_floor, 1.0)
snr_map        = power_map / noise_floor                        # ≈1 for pure noise

# Normalise to [0, 1] using the 99.9th percentile as vmax
snr_max        = np.percentile(snr_map, 99.9)
snr_map_scaled = np.clip(snr_map / snr_max, 0.0, 1.0)

print(f'Power array shape     : {power_map.shape}')
print(f'Frequency range shown : {physical_freqs_ghz[freq_mask].min():.4f} – '
      f'{physical_freqs_ghz[freq_mask].max():.4f} GHz')
print(f'SNR 99.9th pct (vmax) : {snr_max:.2f}')

## 7. Waterfall Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# pcolormesh needs edge arrays one element longer than the data
t_edges = np.append(seg_times_s - seg_dt_s / 2, seg_times_s[-1] + seg_dt_s / 2)
f_ghz   = physical_freqs_ghz[freq_mask]
df_ghz  = np.abs(f_ghz[1] - f_ghz[0])
f_edges = np.append(f_ghz - df_ghz / 2, f_ghz[-1] + df_ghz / 2)

mesh = ax.pcolormesh(
    t_edges,
    f_edges,
    snr_map_scaled,
    cmap='Blues',
    vmin=0.0,
    vmax=1.0,
    shading='flat',
    rasterized=True,
)

cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label('Scaled signal to noise ratio (linear)', fontsize=11)
cbar.set_ticks(np.linspace(0.1, 1.0, 10))

ax.set_xlabel('Time [s]', fontsize=12)
ax.set_ylabel('Frequency [GHz]', fontsize=12)
ax.set_title(f'Project 8 – Event {TEST_TRACK_IDX}', fontsize=13)

fig.tight_layout()
plt.show()

## 8. Side-by-side Comparison: Noisy vs. Denoised

In [ ]:
noisy_power_map = np.zeros((n_display, n_segments), dtype=np.float32)

for s in range(n_segments):
    start = s * CUTOFF
    end   = start + CUTOFF
    I_raw = track_I[start:end] + noise_model(CUTOFF, NOISE_CONST)
    Q_raw = track_Q[start:end] + noise_model(CUTOFF, NOISE_CONST)
    spec  = fftshift(np.abs(fft(I_raw + 1j * Q_raw)) ** 2)
    noisy_power_map[:, s] = spec[freq_mask]

noisy_floor      = np.median(noisy_power_map, axis=0, keepdims=True)
noisy_floor      = np.where(noisy_floor > 0, noisy_floor, 1.0)
noisy_snr        = noisy_power_map / noisy_floor
noisy_snr_scaled = np.clip(noisy_snr / np.percentile(noisy_snr, 99.9), 0.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

for ax, data, title in [
    (axes[0], noisy_snr_scaled, 'Noisy input'),
    (axes[1], snr_map_scaled,   'Denoised (combined model)'),
]:
    mesh = ax.pcolormesh(
        t_edges, f_edges, data,
        cmap='Blues', vmin=0.0, vmax=1.0,
        shading='flat', rasterized=True,
    )
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label('Scaled signal to noise ratio (linear)', fontsize=10)
    cbar.set_ticks(np.linspace(0.1, 1.0, 10))
    ax.set_xlabel('Time [s]', fontsize=12)
    ax.set_ylabel('Frequency [GHz]', fontsize=12)
    ax.set_title(f'Project 8 – Event {TEST_TRACK_IDX} ({title})', fontsize=12)

fig.tight_layout()
plt.show()